# DBSCAN, OPTICS & HDBSCAN — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def pairwise_sq(A,B):
    A=np.asarray(A,float); B=np.asarray(B,float)
    return ((A[:,None,:]-B[None,:,:])**2).sum(axis=2)
def softmax(z):
    z=np.asarray(z,float); z=z-np.max(z)
    e=np.exp(z); return e/e.sum()
def entropy(p):
    p=np.asarray(p,float); p=p[p>0]
    return float(-(p*np.log(p)).sum())
def center(K):
    n=K.shape[0]; H=np.eye(n)-np.ones((n,n))/n
    return H@K@H
print('setup ok')

## One focused concept: the core computation inside DBSCAN, OPTICS & HDBSCAN
These five cells isolate the base calculation, a knob turn, a contrast, an edge case, and a tiny end-to-end pass for DBSCAN, OPTICS & HDBSCAN.

In [ ]:
# 1) nearest representative: the assignment step used by DBSCAN, OPTICS & HDBSCAN
X=np.array([[1.,1.],[1.,2.],[4.,4.],[5.,4.]])
C=np.array([[1.,1.5],[4.5,4.]])
D=pairwise_sq(X,C); assign=D.argmin(axis=1)
plt.figure(figsize=(5,4)); plt.scatter(X[:,0],X[:,1],c=assign,cmap='coolwarm',s=70); plt.scatter(C[:,0],C[:,1],marker='X',c='k',s=160); plt.title('DBSCAN, OPTICS & HDBSCAN: nearest representative')
assert D[1,0]==0.25 and int(assign[2])==1

In [ ]:
# 2) update a representative by averaging its assigned points
X=np.array([[1.,1.],[1.,2.],[4.,4.],[5.,4.]])
assign=np.array([0,0,1,1]); newC=np.vstack([X[assign==k].mean(axis=0) for k in [0,1]])
plt.figure(figsize=(5,4)); plt.scatter(X[:,0],X[:,1],c=assign,cmap='coolwarm',s=70); plt.scatter(newC[:,0],newC[:,1],marker='X',c='k',s=160); plt.title('DBSCAN, OPTICS & HDBSCAN: representative update')
assert np.allclose(newC,[[1,1.5],[4.5,4]])

In [ ]:
# 3) inertia falls after the update
X=np.array([[1.,1.],[1.,2.],[4.,4.],[5.,4.]])
C0=np.array([[0.,0.],[6.,5.]]); C1=np.array([[1.,1.5],[4.5,4.]])
inertia0=pairwise_sq(X,C0).min(axis=1).sum(); inertia1=pairwise_sq(X,C1).min(axis=1).sum()
plt.figure(figsize=(5,3)); plt.bar(['before','after'],[inertia0,inertia1]); plt.title('DBSCAN, OPTICS & HDBSCAN: objective improves')
assert abs(inertia1-1.0)<1e-9 and inertia1<inertia0

In [ ]:
# 4) a scale/threshold knob changes how many neighborhoods look connected
X=np.array([[1.,1.],[1.,2.],[4.,4.],[5.,4.]])
D=pairwise_sq(X,X); eps=np.linspace(.4,4,40); counts=[int(((D>0)&(D<=e**2)).sum()) for e in eps]
plt.figure(figsize=(5,3)); plt.plot(eps,counts); plt.xlabel('neighborhood radius'); plt.ylabel('directed neighbor links'); plt.title('DBSCAN, OPTICS & HDBSCAN: connectivity knob')
assert counts[-1]>counts[0]

In [ ]:
# 5) end-to-end tiny partition with two compact groups
X=np.array([[1.,1.],[1.,2.],[4.,4.],[5.,4.],[4.5,5.]])
C=np.array([[1.,1.5],[4.5,4.3]])
for _ in range(4):
    a=pairwise_sq(X,C).argmin(axis=1); C=np.vstack([X[a==k].mean(axis=0) for k in [0,1]])
plt.figure(figsize=(5,4)); plt.scatter(X[:,0],X[:,1],c=a,cmap='coolwarm',s=70); plt.scatter(C[:,0],C[:,1],marker='X',c='k',s=160); plt.title('DBSCAN, OPTICS & HDBSCAN: converged toy result')
assert set(a)=={0,1} and np.isfinite(C).all()